In [13]:
import jax
import subprocess

def get_gpu_memory_usage_mb():
    try:
        devices = jax.devices()
        if not devices or devices[0].platform != "gpu":
            return 0

        logical_id = devices[0].id  

        cuda_visible = os.environ.get("CUDA_VISIBLE_DEVICES", None)
        if cuda_visible:
            physical_ids = [int(x) for x in cuda_visible.split(",")]
            physical_id = physical_ids[logical_id]
        else:
            physical_id = logical_id

        result = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
            encoding='utf-8'
        )
        mem_values = [int(x) for x in result.strip().split('\n')]

        return mem_values[physical_id]
    except Exception as e:
        print("[get_gpu_memory_usage_mb] get_gpu_memory failed：", e)
        return 0


In [14]:
import os
# 选择GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '4'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.chdir('/home/zheling/Character_Recognition')

print("当前工作目录：", os.getcwd())


当前工作目录： /home/zheling/Character_Recognition


In [15]:
import os
import time
import struct
import json
import math
import random
from functools import partial
from math import sqrt, atan2
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import jax
import jax.numpy as jnp
import jax.nn as jnn
import jax.random as jr
import jax.lax as lax
import jax.tree_util
import optax
import equinox as eqx

from tqdm import tqdm
import signax
from signax import signature
import matplotlib.pyplot as plt


In [16]:
def _concat_points(strokes):
    if len(strokes) == 0:
        return np.zeros((0, 2), dtype=np.float32), []
    idx = []
    all_pts = []
    start = 0
    for s in strokes:
        s = np.asarray(s, dtype=np.float32)
        L = len(s)
        if L == 0:
            continue
        all_pts.append(s)
        idx.append((start, start + L))
        start += L

    if not all_pts:
        return np.zeros((0, 2), dtype=np.float32), []

    return np.concatenate(all_pts, axis=0), idx

def _rotate_pts_about_anchor(P, theta, anchor):
    c, s = np.cos(theta), np.sin(theta)
    xy = P[:, :2]
    anc_xy = anchor[:2]

    D = xy - anc_xy
    x_new = c * D[:, 0] - s * D[:, 1]
    y_new = s * D[:, 0] + c * D[:, 1]

    R = np.stack([x_new, y_new], axis=1) + anc_xy

    if P.shape[1] > 2:
        return np.concatenate([R, P[:, 2:]], axis=1)
    return R

def apply_hanging_on_strokes(strokes, mode="SC"):
    """
    Hanging Normalization
    strokes: List[List[x, y, ...]]
    """
    if (mode is None) or (mode == "None") or (len(strokes) == 0):
        return strokes

    P, spans = _concat_points(strokes)
    if P.shape[0] < 2:
        return strokes

    first_pt = P[0]

    # SC 
    if mode == "SC":
        center = P[:, :2].mean(axis=0)
        v = center - first_pt[:2]
        angle = np.arctan2(v[1], v[0]) + np.pi / 2.0
        P2 = _rotate_pts_about_anchor(P, -angle, first_pt)
        return [P2[s:e] for (s, e) in spans]

    # SE 
    elif mode == "SE":
        last_pt = P[-1]
        v = last_pt[:2] - first_pt[:2]
        angle = np.arctan2(v[1], v[0]) + np.pi / 2.0

        P2 = _rotate_pts_about_anchor(P, -angle, first_pt)
        return [P2[s:e] for (s, e) in spans]

    # ASE 
    elif mode == "ASE":
        out = []
        for s in strokes:
            s_arr = np.asarray(s, dtype=np.float32)
            if len(s_arr) < 2:
                out.append(s_arr)
                continue

            start = s_arr[0]
            end = s_arr[-1]
            v = end[:2] - start[:2]

            angle = np.arctan2(v[1], v[0]) + np.pi / 2.0
            s_rot = _rotate_pts_about_anchor(s_arr, -angle, start)
            out.append(s_rot)
        return out
    else:
        return strokes

In [17]:
def read_pot_file(filename):
    if not os.path.exists(filename):
        return [], ""
    raw_strokes = []
    with open(filename, "rb") as f:
        while True:
            sample_size = f.read(2)
            if sample_size == b'':
                break

            dword_code = f.read(2)
            if len(dword_code) < 2:
                break
            
            if dword_code[0] != 0:
                dword_code = bytes((dword_code[1], dword_code[0]))
            tag_code = struct.unpack(">H", dword_code)[0]

            skip = f.read(2)
            try:
                tag = struct.pack('>H', tag_code).decode("gbk")[0]
            except Exception:
                tag = str(tag_code)

            f.read(2)

            strokes_samples = []
            stroke_samples = []

            while True:
                bx = f.read(2)
                by = f.read(2)
                if bx == b'' or by == b'':
                    if stroke_samples:
                        strokes_samples.append(stroke_samples)
                        stroke_samples = []
                    break
                if bx == b'\xff\xff' and by == b'\xff\xff':
                    if stroke_samples:
                        strokes_samples.append(stroke_samples)
                        stroke_samples = []
                    break
                if bx == b'\xff\xff' and by == b'\x00\x00':
                    strokes_samples.append(stroke_samples)
                    stroke_samples = []
                    continue
                x = struct.unpack("<H", bx)[0]
                y = struct.unpack("<H", by)[0]
                stroke_samples.append((x, y))
            raw_strokes.append(strokes_samples)
    return raw_strokes, tag

def normalize_strokes(strokes, coord_range=1.0):
    if not strokes:
        return strokes
    all_points = [p for s in strokes for p in s]
    if not all_points:
        return strokes
    arr = np.array([np.asarray(p[:2], dtype=np.float32) for p in all_points])
    min_xy, max_xy = arr.min(0), arr.max(0)
    scale = max(max_xy[0] - min_xy[0], max_xy[1] - min_xy[1], 1e-6)

    out = []
    for s in strokes:
        new_s = []
        for p in s:
            extra = tuple(p[2:]) if len(p) > 2 else tuple()
            x = (p[0] - min_xy[0]) / scale * coord_range
            y = (p[1] - min_xy[1]) / scale * coord_range
            new_s.append((x, y) + extra)
        out.append(new_s)
    return out

def simplify_strokes(strokes, dist_thresh=5, angle_thresh=0.15):
    """RDP-like simplification."""
    out = []
    for s in strokes:
        if len(s) <= 2:
            out.append(s)
            continue
        simp = [s[0]]
        for i in range(1, len(s) - 1):
            prev, curr, next_p = simp[-1], s[i], s[i + 1]
            d1 = (curr[0] - prev[0], curr[1] - prev[1])
            d2 = (next_p[0] - curr[0], next_p[1] - curr[1])
            dist_sq = d1[0] ** 2 + d1[1] ** 2
            if d1 == (0, 0) or d2 == (0, 0):
                ang = 0.0
            else:
                ang = abs(math.atan2(d2[1], d2[0]) - math.atan2(d1[1], d1[0]))
            if dist_sq > dist_thresh ** 2 or ang > angle_thresh:
                simp.append(curr)
        simp.append(s[-1])
        out.append(simp)
    return out

def speed_normalization(strokes, target_speed=10.0):
    out = []
    for s in strokes:
        if len(s) <= 1:
            out.append(s)
            continue
        new_s = [s[0]]
        acc_dist = 0.0
        p1 = s[0]
        for i in range(1, len(s)):
            p2 = s[i]
            dist = math.hypot(p2[0] - p1[0], p2[1] - p1[1])
            acc_dist += dist
            curr_p1 = p1
            curr_dist = dist
            while acc_dist >= target_speed:
                acc_dist -= target_speed
                if curr_dist > 1e-6:
                    t = 1 - (acc_dist / curr_dist)
                    extra = tuple(curr_p1[2:]) if len(curr_p1) > 2 else tuple()
                    new_pt = (curr_p1[0] + t * (p2[0] - curr_p1[0]),
                              curr_p1[1] + t * (p2[1] - curr_p1[1])) + extra
                    new_s.append(new_pt)
                    curr_p1 = new_pt
                    # after interpolation, remaining distance becomes acc_dist which was carried
                    curr_dist = acc_dist
                else:
                    break
            p1 = p2
        new_s.append(s[-1])
        out.append(new_s)
    return out

def add_imaginary_stroke(strokes, add_ink_dim=True, add_pen_states=False):
    if len(strokes) < 2:
        return [{"data": np.array(s, dtype=np.float32), "type": "real"} for s in strokes]

    total_len = 0.0
    count = 0
    for s in strokes:
        for i in range(len(s) - 1):
            total_len += math.hypot(s[i + 1][0] - s[i][0], s[i + 1][1] - s[i][1])
            count += 1
    avg_len = total_len / max(1, count)

    out = []
    for i in range(len(strokes)):
        out.append({"data": np.array(strokes[i], dtype=np.float32), "type": "real"})
        if i < len(strokes) - 1:
            start = strokes[i][-1]
            end = strokes[i + 1][0]
            dist = math.hypot(end[0] - start[0], end[1] - start[1])
            steps = max(1, int(math.ceil(dist / max(1e-6, avg_len))))
            v_stroke = []
            for j in range(steps + 1):
                alpha = j / steps
                extra = tuple(start[2:]) if len(start) > 2 else tuple()
                pt = (start[0] * (1 - alpha) + end[0] * alpha,
                      start[1] * (1 - alpha) + end[1] * alpha) + extra
                v_stroke.append(pt)
            out.append({"data": np.array(v_stroke, dtype=np.float32), "type": "virtual"})
    return out

def apply_data_augmentation(strokes, key):
    if not strokes:
        return strokes
    k_s, k_t, k_e = jr.split(key, 3)
    sx = 1.0 + float(jr.uniform(k_s, minval=-0.15, maxval=0.15))
    sy = 1.0 + float(jr.uniform(k_s, minval=-0.15, maxval=0.15))
    tx = float(jr.normal(k_t)) * 10.0
    ty = float(jr.normal(k_t)) * 10.0
    eps = float(jr.uniform(k_e, minval=0.0, maxval=2.0))

    all_pts = []
    for s in strokes:
        for p in s:
            all_pts.append(p[:2])

    if len(all_pts) == 0:
        return strokes
    arr = np.array(all_pts, dtype=np.float32)

    w, h = (arr.max(0) - arr.min(0)) + 1e-6
    fx, fy = 2 * math.pi / h, 2 * math.pi / w

    aug_strokes = []
    for s in strokes:
        new_stroke = []
        for p in s:
            x = p[0] * sx + tx + eps * math.sin(p[1] * sy * fx)
            y = p[1] * sy + ty + eps * math.sin(p[0] * sx * fy)
            new_pt = (x, y) + tuple(p[2:])
            new_stroke.append(new_pt)
        aug_strokes.append(new_stroke)
    return aug_strokes

def apply_rotation_on_strokes(strokes, angle_deg):
    """Rotates the entire character around the origin (0,0) by a specified angle."""
    if angle_deg is None or angle_deg == 0:
        return strokes
    theta = np.deg2rad(float(angle_deg))
    c, s = np.cos(theta), np.sin(theta)
    out = []
    for stroke in strokes:
        s_arr = np.array(stroke, dtype=np.float32)
        if len(s_arr) == 0:
            out.append(s_arr)
            continue
        x, y = s_arr[:, 0], s_arr[:, 1]
        xr = c * x - s * y
        yr = s * x + c * y
        if s_arr.shape[1] > 2:
            new_s = np.concatenate([np.stack([xr, yr], 1), s_arr[:, 2:]], 1)
        else:
            new_s = np.stack([xr, yr], 1)
        # convert back to list of tuples for consistency with other functions
        out.append([tuple(row) for row in np.asarray(new_s)])
    return out

def compute_derivatives_strokes(strokes):
    """
    Computes differential features for strokes based on differences.
    Output format per point: [x, y, (ink...), dx, dy, dxx, dyy]
    """
    seq = []
    for stroke in strokes:
        s_arr = np.array(stroke, dtype=np.float32)
        n = len(s_arr)
        if n == 0:
            continue
        xy = s_arr[:, :2]

        vd = np.zeros_like(xy)
        vc = np.zeros_like(xy)

        if n >= 2:
            vd[0] = xy[1] - xy[0]
            vd[-1] = xy[-1] - xy[-2]
            if n > 2:
                vd[1:-1] = xy[2:] - xy[:-2]

            vc[0] = vd[1] - vd[0] if n > 1 else 0
            vc[-1] = vd[-1] - vd[-2] if n > 1 else 0
            if n > 2:
                vc[1:-1] = vd[2:] - vd[:-2]
        feat = np.concatenate([s_arr, vd, vc], axis=1)
        for point in feat:
            seq.append(point)
    return np.array(seq, dtype=np.float32)

def build_feature_sequence(input_strokes, add_ink_dim, add_pen_states, use_derivatives):
    if len(input_strokes) > 0 and isinstance(input_strokes[0], dict):
        stroke_objs = input_strokes
    else:
        stroke_objs = [{"data": np.array(s, dtype=np.float32), "type": "real"} for s in input_strokes]

    processed = []
    for item in stroke_objs:
        s, stype = item["data"], item["type"]
        xy = s[:, :2]

        if add_ink_dim:
            if stype == 'real':
                if s.shape[1] >= 3:
                    ink = s[:, 2:3]
                else:
                    ink = np.ones((len(s), 1), dtype=np.float32)
            else:
                ink = np.zeros((len(s), 1), dtype=np.float32)
            xy = np.concatenate([xy, ink], axis=1)

        if add_pen_states:
            pd = np.zeros((len(s), 1), dtype=np.float32)
            pu = np.zeros((len(s), 1), dtype=np.float32)
            if stype == 'real' and len(s) > 0:
                pd[0] = 1.0
                pu[-1] = 1.0
            xy = np.concatenate([xy, pd, pu], axis=1)
        processed.append(xy)

    if not processed:
        return np.zeros((0, 2), dtype=np.float32)

    if use_derivatives:
        final_seq = []
        for feat in processed:
            coords = feat[:, :2]
            n = len(coords)
            vd = np.zeros_like(coords)
            vc = np.zeros_like(coords)
            if n >= 2:
                vd[0] = coords[1] - coords[0]
                vd[-1] = coords[-1] - coords[-2]
                if n > 2:
                    vd[1:-1] = (coords[2:] - coords[:-2]) / 2.0
                vc[0] = vd[1] - vd[0]
                vc[-1] = vd[-1] - vd[-2]
                if n > 2:
                    vc[1:-1] = (vd[2:] - vd[:-2]) / 2.0
            final_seq.append(np.concatenate([feat, vd, vc], axis=1))
        return np.concatenate(final_seq, 0)

    return np.concatenate(processed, 0)

def normalize_all_dimensions(sequence, add_ink_dim, add_pen_states):
    if len(sequence) == 0:
        return sequence
    seq = sequence.copy()

    current_idx = 2
    if add_ink_dim:
        max_ink = seq[:, 2].max()
        if max_ink > 0:
            seq[:, 2] /= max_ink
        current_idx += 1

    if add_pen_states:
        current_idx += 2

    if current_idx < seq.shape[1]:
        derivs = seq[:, current_idx:]
        max_val = np.max(np.abs(derivs))
        if max_val > 1e-6:
            seq[:, current_idx:] /= max_val

    return seq

def process_sequence_length(seq, target_len, use_end_pad):
    L, D = seq.shape
    if L >= target_len:
        return seq[:target_len]
    out = np.zeros((target_len, D), dtype=np.float32)
    out[:L] = seq
    if use_end_pad and L > 0:
        out[L:] = seq[-1]
        if D >= 4:
            out[L:, -4:] = 0
    return out

@eqx.filter_jit
def compute_window_signatures(path, window_size=10, stride=1, depth=2):
    """
    Computes Path Signatures using sliding windows.
    """
    length, dim = path.shape
    if length < window_size:
        window_size = length
    num_windows = max(1, (length - window_size) // stride + 1)

    starts = jnp.arange(num_windows) * stride
    indices = starts[:, None] + jnp.arange(window_size)[None, :]
    indices = jnp.clip(indices, 0, length - 1)

    windows = path[indices]

    sigs = jax.vmap(lambda w: signax.signature(w, depth=depth))(windows)
    return sigs

def build_one_feature(strokes, cfg, key_aug=None, angle=None):
    if cfg.get('use_simplify', False):
        strokes = simplify_strokes(strokes, cfg.get('dist_thresh', 5), cfg.get('angle_thresh', 0.15))
    if cfg.get('use_speed_norm', False):
        strokes = speed_normalization(strokes, cfg.get('target_speed', 10.0))
    if angle:
        strokes = apply_rotation_on_strokes(strokes, angle)

    if key_aug is not None:
        strokes = apply_data_augmentation(strokes, key_aug)

    if cfg.get('use_hanging', False):
        strokes = apply_hanging_on_strokes(strokes, cfg.get('hanging_mode', "SC"))

    strokes = normalize_strokes(strokes, cfg.get('coord_range', 1.0))

    if cfg.get('add_imaginary_stroke', False) and len(strokes) > 1:
        strokes_data = add_imaginary_stroke(strokes, cfg.get('add_ink_dim', False), cfg.get('add_pen_states', False))
    else:
        strokes_data = [{"data": np.array(s, dtype=np.float32), "type": "real"} for s in strokes]

    seq = build_feature_sequence(strokes_data, cfg.get('add_ink_dim', False), cfg.get('add_pen_states', False), cfg.get('use_derivatives', False))
    seq = normalize_all_dimensions(seq, cfg.get('add_ink_dim', False), cfg.get('add_pen_states', False))

    seq = process_sequence_length(seq, cfg.get('target_length', 100), cfg.get('use_end_padding', True))

    if cfg.get('use_signatures', False):
        return compute_window_signatures(jnp.array(seq), cfg.get('window_size', 10), cfg.get('stride', 1), cfg.get('depth', 2))
    return seq

In [18]:
class ArrayDataloader:
    def __init__(self, X, y, a=None, batch_size=64):
        self.X = jnp.asarray(X)
        self.y = jnp.asarray(y)
        self.a = jnp.asarray(a) if a is not None else None
        self.size = len(X)
        self.batch_size = batch_size  
        self.datas = self.X
        self.labels = self.y
        self.angles = self.a if self.a is not None else jnp.zeros((self.size,))

    def loop(self, bs, key, shuffle=True):
        idx = jr.permutation(key, jnp.arange(self.size)) if shuffle else jnp.arange(self.size)
        for i in range(0, self.size, bs):
            b = idx[i:i + bs]
            yield self.X[b], self.y[b], (self.a[b] if self.a is not None else None)


def process_class(c_idx, data_dir, is_train, cfg, seed=None):
    raw, _ = read_pot_file(os.path.join(data_dir, f"{c_idx}.pot"))
    if not raw:
        return [], [], []  

    n_tr = int(0.8 * len(raw))
    sel = raw[:n_tr] if is_train else raw[n_tr:]

    if is_train:
        feats, labels, angles = [], [], []
        key = jr.PRNGKey(int(seed) if seed is not None else 42)
        for i, s in enumerate(sel):
            k1, k2 = jr.split(jr.fold_in(key, i))
            ang = float(jr.uniform(k1, minval=0, maxval=360))
            f = build_one_feature(s, cfg, key_aug=k2, angle=ang)
            time = np.linspace(0, 1, f.shape[0], dtype=np.float32)
            f_with_time = np.column_stack([time, f])
            
            feats.append(f_with_time)
            labels.append(c_idx - 1)
            angles.append(ang)
        return feats, labels, angles
    else:
        f30, l30, a30 = [], [], []
        ang_list = list(range(0, 360, 12))
        for i, s in enumerate(sel):
            for ang in ang_list:
                f = build_one_feature(s, cfg, angle=float(ang))
                time = np.linspace(0, 1, f.shape[0], dtype=np.float32)
                f_with_time = np.column_stack([time, f])
                
                f30.append(f_with_time)
                l30.append(c_idx - 1)
                a30.append(float(ang))
        return f30, l30, a30


def make_loaders(data_dir, n_cls, cfg, seed=42, mode="train"):
    tr, te30 = ([], [], []), ([], [], [])

    for c in tqdm(range(1, n_cls + 1), desc=f"Build {mode} set"):
        if mode == "train":
            F, L, A = process_class(c, data_dir, True, cfg, seed)
            tr[0].extend(F)
            tr[1].extend(L)
            tr[2].extend(A)

        f30, l30, a30 = process_class(c, data_dir, False, cfg)
        te30[0].extend(f30)
        te30[1].extend(l30)
        te30[2].extend(a30)

    def _L(d):
        return ArrayDataloader(d[0], d[1], d[2])

    if mode == "train":
        tr_loader = _L(tr)
        te30_loader = _L(te30)
        dim = tr_loader.X.shape[2] if len(tr_loader.X) > 0 else None
        
        print(f"Train samples: {len(tr[0])}, shape: {tr_loader.X.shape}")
        print(f"Test30 samples: {len(te30[0])}, shape: {te30_loader.X.shape}")
        print(f"Feature dimension: {dim}")
        
        return tr_loader, te30_loader, dim
    else:
        te30_loader = _L(te30)
        dim = te30_loader.X.shape[2] if len(te30[0]) > 0 else None
        
        print(f"Test30 samples: {len(te30[0])}, shape: {te30_loader.X.shape}")
        print(f"Feature dimension: {dim}")
        
        return te30_loader, dim


In [19]:

import functools
import itertools
from collections import defaultdict
from typing import Generator, Union

import jax.numpy as jnp
import numpy as np
from scipy.sparse import csc_matrix


def tkey_to_index(width: int, tkey: Union[int, tuple[int]]) -> int:
    if isinstance(tkey, int):
        assert tkey <= width
        return tkey

    result = 0
    for letter in tkey:
        result *= width
        result += letter
    return result


def tensor_algebra_dimension(width: int, depth: int) -> int:
    result = 1
    for _ in range(depth):
        result *= width
        result += 1
    return result


def generate_tensor_keys_level(
    width: int, degree: int
) -> Generator[tuple[int], None, None]:
    if degree == 1:
        yield from ((i,) for i in range(1, width + 1))
        return

    for i in range(1, width + 1):
        yield from ((i, *r) for r in generate_tensor_keys_level(width, degree - 1))


def generate_tensor_keys(width: int, depth: int) -> Generator[tuple[int], None, None]:
    yield ()

    if depth == 0:
        return

    for i in range(1, width + 1):
        yield (i,)

    for degree in range(2, depth + 1):
        yield from generate_tensor_keys_level(width, degree)

class HallSet:
    def __init__(self, width, degree=1):
        self.width = width
        self.degree = 1

        self.data = data = []
        self.reverse_map = reverse_map = {}
        self.degree_ranges = degree_ranges = []
        self.sizes = sizes = []
        self.letters = letters = []
        self.l2k = l2k = {}

        data.append((0, 0))
        degree_ranges.append((0, 1))
        sizes.append(0)

        for letter in range(1, width + 1):
            parents = (0, letter)
            letters.append(letter)
            data.append(parents)
            reverse_map[parents] = letter
            l2k[letter] = letter

        degree_ranges.append((degree_ranges[0][1], len(data)))
        sizes.append(width)

        if degree > self.degree:
            self.grow_up(degree)

    def grow_up(self, degree):

        data = self.data
        reverse_map = self.reverse_map
        degree_ranges = self.degree_ranges

        while self.degree < degree:
            next_degree = self.degree + 1
            left = 1
            while 2 * left <= next_degree:
                right = next_degree - left

                ilower, iupper = degree_ranges[left]
                jlower, jupper = degree_ranges[right]

                i = ilower

                while i < iupper:
                    j = max(jlower, i + 1)
                    while j < jupper:
                        if data[j][0] <= i:
                            parents = (i, j)
                            data.append(parents)
                            reverse_map[parents] = len(data) - 1
                        j += 1
                    i += 1
                left += 1

            degree_ranges.append((degree_ranges[-1][1], len(data)))
            self.sizes.append(len(data))
            self.degree += 1

    @functools.lru_cache
    def key_to_string(self, key: int) -> str:
        assert key < len(self.data)

        left, right = self.data[key]

        if left == 0:
            return f"{right}"

        return f"[{self.key_to_string(left)}, {self.key_to_string(right)}]"

    @functools.lru_cache
    def product(self, lhs_key: int, rhs_key: int) -> list[tuple[int, int]]:
        if rhs_key < lhs_key:
            return [(k, -c) for k, c in self.product(rhs_key, lhs_key)]

        if lhs_key == rhs_key:
            return []

        if key := self.reverse_map.get((lhs_key, rhs_key)):
            return [(key, 1)]

        lparent, rparent = self.data[rhs_key]

        left_result = [
            (k, c1 * c)
            for (k1, c1) in self.product(lhs_key, lparent)
            for (k, c) in self.product(k1, rparent)
        ]
        right_result = [
            (k, -c1 * c)
            for (k1, c1) in self.product(lhs_key, rparent)
            for (k, c) in self.product(k1, lparent)
        ]
        result = defaultdict(lambda: 0)
        for k, c in left_result:
            result[k] += c
        for k, c in right_result:
            result[k] += c

        return list(result.items())

    @functools.lru_cache
    def expand(self, key: int) -> list[tuple[int, tuple[int]]]:
        if key in self.letters:
            return [((key,), 1)]

        assert key < len(self.data)
        lparent, rparent = self.data[key]

        left_expansion = self.expand(lparent)
        right_expansion = self.expand(rparent)

        left_terms = [
            ((*k1, *k2), c1 * c2)
            for (k1, c1), (k2, c2) in itertools.product(left_expansion, right_expansion)
        ]
        right_terms = [
            ((*k1, *k2), c1 * c2)
            for (k1, c1), (k2, c2) in itertools.product(right_expansion, left_expansion)
        ]

        result = defaultdict(lambda: 0)
        for k, c in left_terms:
            result[k] += c
        for k, c in right_terms:
            result[k] -= c

        return list(result.items())

    @functools.lru_cache
    def rbracket(self, tkey: Union[int, tuple[int]]) -> list[tuple[int, int]]:
        if isinstance(tkey, int):
            return [(tkey, 1)]

        if len(tkey) == 0:
            return []

        if len(tkey) == 1:
            return [(tkey[0], 1)]

        assert len(tkey) > 1, f"{tkey}"
        first, *remaining = tkey
        return [
            (k, c1 * c)
            for (k1, c1) in self.rbracket(tuple(remaining))
            for k, c in self.product(first, k1)
        ]

    def l2t_matrix(self, degree=None, dtype=np.float32) -> jnp.ndarray:
        degree = degree or self.degree
        tensor_alg_size = tensor_algebra_dimension(self.width, degree)

        indptr = [0, 0]
        indices = []
        data = []
        for lkey in range(1, self.sizes[degree]):
            for k, c in self.expand(lkey):
                indices.append(tkey_to_index(self.width, k))
                data.append(c)
            indptr.append(indptr[-1] + len(self.expand(lkey)))

        data = np.array(data, dtype=dtype)
        indices = np.array(indices, dtype=np.int64)
        indptr = np.array(indptr, dtype=np.int64)
        return jnp.array(
            csc_matrix(
                (data, indices, indptr),
                shape=(tensor_alg_size, self.sizes[degree]),
                dtype=dtype,
            ).toarray()
        )

    def t2l_matrix(self, degree=None, dtype=np.float32) -> jnp.ndarray:
        degree = degree or self.degree
        tensor_alg_size = tensor_algebra_dimension(self.width, degree)

        indptr = [0]
        indices = []
        data = []
        for tkey in generate_tensor_keys(self.width, degree):
            for k, c in self.rbracket(tkey):
                indices.append(k)
                data.append(c / len(tkey))
            indptr.append(len(data))
        data = np.array(data, dtype=dtype)
        indices = np.array(indices, dtype=np.int64)
        indptr = np.array(indptr, dtype=np.int64)

        return jnp.array(
            csc_matrix(
                (data, indices, indptr),
                shape=(self.sizes[degree], tensor_alg_size),
                dtype=dtype,
            ).toarray()
        )

In [20]:
import equinox as eqx
import jax
import jax.numpy as jnp
import jax.nn as jnn
import jax.random as jr
import diffrax

class NeuralCDE(eqx.Module):
    vf: eqx.nn.MLP  
    data_dim: int 
    hidden_dim: int  
    ode_solver_stepsize: int  
    linear1: eqx.nn.Linear  
    linear2: eqx.nn.Linear  
    linear3: eqx.nn.Linear 
    
    def __init__(
        self,
        hidden_dim,
        data_dim,
        label_dim,
        vf_hidden_dim,
        vf_num_hidden,
        ode_solver_stepsize,
        *,
        key,
    ):
        vf_key, l1key, l2key, l3key = jr.split(key, 4)

        self.vf = eqx.nn.MLP(
            hidden_dim,
            hidden_dim * data_dim,
            vf_hidden_dim,
            vf_num_hidden,
            activation=jax.nn.silu,
            final_activation=jax.nn.tanh,
            key=vf_key,
        )

        self.linear1 = eqx.nn.Linear(data_dim, hidden_dim, key=l1key)
        self.linear2 = eqx.nn.Linear(hidden_dim, label_dim, key=l2key)
        self.linear3 = eqx.nn.Linear(hidden_dim, 2, key=l3key)
        self.hidden_dim = hidden_dim
        self.data_dim = data_dim
        self.ode_solver_stepsize = ode_solver_stepsize

    def get_ode(self, ts, X):
        coeffs = diffrax.backward_hermite_coefficients(ts, X)
        control = diffrax.CubicInterpolation(ts, coeffs)
        func = lambda t, y, args: jnp.reshape(
            self.vf(y), (self.hidden_dim, self.data_dim)
        )
        return diffrax.ControlTerm(func, control).to_ode()


    def __call__(self, X):
        ts = X[:, 0]
        ode_term = self.get_ode(ts, X)
        h0 = self.linear1(X[0, :])

        saveat = diffrax.SaveAt(t1=True)
        solution = diffrax.diffeqsolve(
            terms=ode_term,
            solver=diffrax.Heun(), 
            t0=ts[0],
            t1=ts[-1],
            dt0=self.ode_solver_stepsize,
            y0=h0,
            saveat=saveat,
            stepsize_controller=diffrax.ConstantStepSize(),
        )

        x = solution.ys[-1]
        x = self.linear2(x)

        return x


In [21]:
import jax
import jax.numpy as jnp
import diffrax
import equinox as eqx
from jax import vmap, jacrev
from functools import partial
from signax import signature
from signax.tensor_ops import log

class LogNeuralCDE(NeuralCDE):
    stepsize: int  
    depth: int  
    hall_set: HallSet  
    
    def __init__(self,
                 hidden_dim,
                 data_dim,
                 label_dim,
                 vf_hidden_dim,
                 vf_num_hidden,
                 stepsize,
                 depth,
                 *,
                 key,
                 **kwargs):

        super().__init__(
            hidden_dim,
            data_dim,
            label_dim,
            vf_hidden_dim,
            vf_num_hidden,
            key=key,
            **kwargs
        )

        self.stepsize = stepsize
        if depth not in [1, 2]:
            raise ValueError("The Log-ODE method is only implemented for depth 1 or 2")
        self.depth = depth
        self.hall_set = HallSet(data_dim, depth)

        
    def calc_logsigs(self, X):
        X = X.reshape(-1, self.stepsize, X.shape[-1])  # (batchsize, stepsize, feadim)
        prepend = jnp.concatenate((jnp.zeros((1, X.shape[-1])), X[:-1, -1, :]))[:, None, :]
        X = jnp.concatenate((prepend, X), axis=1)
    
        def logsig(x):
            sig = jnp.array(signature(x, self.depth))  
            log_sig = jnp.array(log(sig))  
            flat_logsig = log_sig.reshape(-1)  
            
            if self.depth == 1:
                return jnp.concatenate((jnp.array([0.]), flat_logsig))
            else:
                tensor_to_lie_map = self.hall_set.t2l_matrix(self.depth)
                return tensor_to_lie_map[:, 1:] @ flat_logsig
    
        return jax.vmap(logsig)(X)


    def depth_one_ode(self, y, logsig, interval_length):
        vf_out = jnp.reshape(self.vf(y), (self.hidden_dim, self.data_dim))
        return jnp.dot(vf_out, logsig[1:]) / interval_length

    def depth_two_ode(self, y, logsig, interval_length):
        vf_out = jnp.reshape(self.vf(y), (self.hidden_dim, self.data_dim))

        jvps = jnp.reshape(
            jax.vmap(lambda x: jax.jvp(self.vf, (y,), (x,))[1])(vf_out.T),
            (self.data_dim, self.data_dim, self.hidden_dim),
        )

        def liebracket(jvps, pair):
            return jvps[pair[0] - 1, pair[1] - 1] - jvps[pair[1] - 1, pair[0] - 1]

        pairs = jnp.asarray(self.hall_set.data[self.data_dim + 1 :])
        lieout = jax.vmap(liebracket, in_axes=(None, 0))(jvps, pairs)

        vf_depth1 = jnp.dot(vf_out, logsig[1 : self.data_dim + 1])
        vf_depth2 = jnp.dot(lieout.T, logsig[self.data_dim + 1 :])

        return (vf_depth1 + vf_depth2) / interval_length

    def get_ode(self, ts, X):

        logsigs = self.calc_logsigs(X)
        intervals = (
            jnp.arange(0, X.shape[0] + self.stepsize, self.stepsize) / X.shape[0]
        )

        def func(t, y, args):
            idx = jnp.searchsorted(intervals, t)
            logsig_t = logsigs[idx - 1]
            interval_length = intervals[idx] - intervals[idx - 1]
            if self.depth == 1:
                return self.depth_one_ode(y, logsig_t, interval_length)
            if self.depth == 2:
                return self.depth_two_ode(y, logsig_t, interval_length)

        return diffrax.ODETerm(func)

In [22]:
@eqx.filter_jit
def train_step(model, x, y, opt, opt_state, subkey):
    def loss_fn(m):
        preds = jax.vmap(m)(x)
        loss = optax.softmax_cross_entropy_with_integer_labels(preds, y).mean()
        acc = jnp.mean(jnp.argmax(preds, axis=-1) == y)
        return loss, acc

    (loss, acc), grads = eqx.filter_value_and_grad(loss_fn, has_aux=True)(model)
    updates, opt_state = opt.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss, acc


def train_epoch(model, loader, opt, opt_state, key):
    losses, accs, steps = [], [], 0
    for x, y, _ in loader.loop(loader.batch_size, key, shuffle=True):
        key, subkey = jr.split(key)
        model, opt_state, loss, acc = train_step(model, x, y, opt, opt_state, subkey)
        losses.append(float(loss))
        accs.append(float(acc))
        steps += 1
    return model, opt_state, np.mean(losses), np.mean(accs), steps


def _get_logits_and_acc(model, loader, batch_size=512, key_seed=123, shuffle=False):
    logits_list, targets_list = [], []
    eval_key = jr.PRNGKey(key_seed)
    for x, y, _ in loader.loop(batch_size, eval_key, shuffle=shuffle):
        preds = jax.vmap(model)(x)
        logits_list.append(np.array(preds))
        targets_list.append(np.array(y))
    all_logits = np.concatenate(logits_list, axis=0)
    all_targets = np.concatenate(targets_list, axis=0)
    acc = np.mean(np.argmax(all_logits, axis=-1) == all_targets)
    return all_logits, all_targets, float(acc)



def train_loop(model, loader, epochs, lr_fn, save_dir, ckpt_every, print_every, key):
    opt = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(lr_fn))
    opt_state = opt.init(eqx.filter(model, eqx.is_inexact_array))
    log, gpu_mem_records  = {"epoch": [], "loss": [], "acc": [], "time": [], "mem": []}, [] 
    total_steps, t0_total = 0, time.time()

    for ep in range(epochs):
        t0_epoch = time.time()
        key, epoch_key = jr.split(key)
        model, opt_state, l, a, steps_in_epoch = train_epoch(model, loader, opt, opt_state, epoch_key)
        total_steps += steps_in_epoch
        cur_mem = get_gpu_memory_usage_mb()
        epoch_time = time.time() - t0_epoch
        gpu_mem_records.append(cur_mem)

        if (ep + 1) % print_every == 0 or ep == 0:
            print(f"Ep {ep+1}/{epochs} | Loss: {l:.4f} | Acc: {a:.2%} | Mem: {cur_mem:.2f}MB | Time：{epoch_time:.2f}s")

        log["epoch"].append(ep + 1)
        log["loss"].append(float(l))
        log["acc"].append(float(a))
        log["time"].append(epoch_time)
        log["mem"].append(cur_mem)

        if save_dir and ckpt_every and (ep + 1) % ckpt_every == 0:
            p = os.path.join(save_dir, "checkpoints")
            os.makedirs(p, exist_ok=True)
            eqx.tree_serialise_leaves(os.path.join(p, f"ep{ep+1}.eqx"), model)
            eqx.tree_serialise_leaves(os.path.join(p, f"opt_state_ep{ep+1}.eqx"), opt_state)
            
    total_time = time.time() - t0_total
    avg_mem = float(np.mean(log["mem"])) if log["mem"] else 0.0
    time_per_1000_steps = (total_time / total_steps * 1000) if total_steps > 0 else 0.0

    stats = {
        "total_time": total_time,
        "total_steps": total_steps,
        "avg_mem": avg_mem,
        "time_per_1000_steps": time_per_1000_steps
    }

    print("\n=== Training Stats ===")
    print(f"Total Time: {total_time:.2f}s")
    print(f"Total Steps: {total_steps}")
    print(f"Time per 1000 steps: {time_per_1000_steps:.2f}s")
    print(f"Avg GPU Memory: {avg_mem:.2f}MB")

    return model, log, stats


In [23]:
def compute_ensemble_metrics(results_list):
    if not results_list:
        return {}
    accs = [r['acc'] for r in results_list]
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)

    all_logits = np.stack([r['logits'] for r in results_list], axis=0)  # [S, N, C]
    targets = results_list[0]['targets']

    # Soft voting
    probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)
    avg_probs = np.mean(probs, axis=0)
    soft_acc = np.mean(np.argmax(avg_probs, -1) == targets)

    # Hard voting
    seed_preds = np.argmax(all_logits, axis=-1)
    hard_preds = [np.argmax(np.bincount(seed_preds[:, i])) for i in range(seed_preds.shape[1])]
    hard_acc = np.mean(np.array(hard_preds) == targets)

    return {"mean": mean_acc, "std": std_acc, "soft": soft_acc, "hard": hard_acc}



def run_single_model_ncde(tr_load, te30_load, data_dim, n_cls, save_dir, cfg, seed):
    if os.path.exists(os.path.join(save_dir, "_COMPLETED")):
        print(f"Skipping {save_dir} (already completed)")
        return None

    os.makedirs(save_dir, exist_ok=True)
    print(f"Data dimension: {data_dim}")

    key = jr.PRNGKey(seed)
    modelkey, trainkey = jr.split(key)

    model = LogNeuralCDE(
        hidden_dim=cfg["hidden_dim"],
        data_dim=data_dim,
        label_dim=n_cls,
        vf_hidden_dim=cfg["vf_hidden_dim"],
        vf_num_hidden=cfg["vf_num_hidden"],
        stepsize=cfg["stepsize"],
        depth=cfg["depth"],
        key=modelkey,
        ode_solver_stepsize=cfg["ode_solver_stepsize"],
    )

    lr = optax.exponential_decay(1e-3, 4500, 0.5, staircase=True, end_value=1e-6)

    model, log, stats = train_loop(
        model, tr_load, cfg["total_epochs"], lr, save_dir,
        cfg["save_ckpt_every"], cfg["print_every"], key=trainkey
    )

    print("\n=== Evaluating on 30° Rotated Test Set ===")
    l30, t30, a30 = _get_logits_and_acc(model, te30_load)
    print(f"30° Test Accuracy: {a30:.4f}")

    eqx.tree_serialise_leaves(os.path.join(save_dir, "model.eqx"), model)

    res_summary = {
        "cfg": cfg,
        "seed": seed,
        "acc30": a30,
        "train_time": stats['total_time'],
        "total_steps": stats['total_steps'],
        "time_per_1000_steps": stats['time_per_1000_steps'],
        "avg_mem": stats['avg_mem']
    }

    with open(os.path.join(save_dir, "summary.json"), "w") as f:
        json.dump(res_summary, f, indent=2)
    open(os.path.join(save_dir, "_COMPLETED"), "w").close()

    return {"summary": res_summary,
            "res_30x": {"logits": l30, "targets": t30, "acc": a30}}



def run_experiment(data_dir, n_cls, save_dir, cfg):
    seeds = cfg.pop('seed_list')

    print(f">>> Single Run. Seeds: {seeds}")
    print("Loading and splitting data (once for all seeds)...")

    tr_load, te30_load, dim = make_loaders(data_dir, n_cls, cfg, seed=42, mode="train")
    data_dim = dim[-1] if isinstance(dim, tuple) else dim
    results = []

    for s in seeds:
        res = run_single_model_ncde(
            tr_load, te30_load, data_dim, n_cls,
            os.path.join(save_dir, f"seed_{s}"), cfg.copy(), s
        )
        if res:
            results.append(res)

    if results:
        m30 = compute_ensemble_metrics([r['res_30x'] for r in results])
        print("\n=== FINAL RESULTS ===")
        print(f"Test 30x: Mean±Std {m30['mean']:.4f} ± {m30['std']:.4f}")
        print(f"Soft Voting: {m30['soft']:.4f}")
        print(f"Hard Voting: {m30['hard']:.4f}")


In [28]:
if __name__ == "__main__":
    DATA_DIR = "RotFreeData/EngUpper26"  # EngUpper26 Digits10
    SAVE_DIR = "Model/Logncde/"
    NUM_CLASSES = 26

    BASE_CONFIG = {
        # Preprocessing
        "use_simplify": True, "dist_thresh": 5.0, "angle_thresh": 0.15,
        "use_speed_norm": True, "target_speed": 10.0,
        "coord_range": 1.0,
        "add_ink_dim": False, "add_pen_states": False, "add_imaginary_stroke": False,
        "use_derivatives": False, "use_end_padding": True, "use_signatures": False,
        "use_hanging": True,
        "hanging_mode": "SC",
        'target_length': 100,
        
        # Train
        "print_every": 20, "save_ckpt_every": 0,
        "total_epochs": 200, "batch_size": 64,
        
        # Model 
        "hidden_dim": 128, "vf_hidden_dim": 256, "vf_num_hidden": 2,
        "ode_solver_stepsize": 1/50,
        "stepsize": 1,
        "depth": 2,
        
        # Seed
        "seed_list": [1011,1012,1013]
    }

    run_experiment(DATA_DIR, NUM_CLASSES, SAVE_DIR, BASE_CONFIG)

>>> Single Run. Seeds: [1011, 1012, 1013]
Loading and splitting data (once for all seeds)...


Build train set: 100%|██████████████████████████████████████████████████████████████████| 26/26 [00:27<00:00,  1.06s/it]


Train samples: 6237, shape: (6237, 100, 3)
Test30 samples: 46800, shape: (46800, 100, 3)
Feature dimension: 3
Data dimension: 3
Ep 1/200 | Loss: 2.5887 | Acc: 24.18% | Mem: 2709.00MB | Time：27.67s
Ep 20/200 | Loss: 0.2007 | Acc: 94.05% | Mem: 2709.00MB | Time：9.45s
Ep 40/200 | Loss: 0.0648 | Acc: 97.94% | Mem: 2709.00MB | Time：9.49s
Ep 60/200 | Loss: 0.0069 | Acc: 99.86% | Mem: 2709.00MB | Time：9.47s
Ep 80/200 | Loss: 0.0078 | Acc: 99.82% | Mem: 2709.00MB | Time：9.48s
Ep 100/200 | Loss: 0.0008 | Acc: 100.00% | Mem: 2709.00MB | Time：9.49s
Ep 120/200 | Loss: 0.0003 | Acc: 100.00% | Mem: 2709.00MB | Time：9.49s
Ep 140/200 | Loss: 0.0114 | Acc: 99.65% | Mem: 2709.00MB | Time：9.49s
Ep 160/200 | Loss: 0.0021 | Acc: 99.98% | Mem: 2709.00MB | Time：9.48s
Ep 180/200 | Loss: 0.0001 | Acc: 100.00% | Mem: 2709.00MB | Time：9.48s
Ep 200/200 | Loss: 0.0001 | Acc: 100.00% | Mem: 2709.00MB | Time：9.49s

=== Training Stats ===
Total Time: 1916.34s
Total Steps: 19600
Time per 1000 steps: 97.77s
Avg GPU Mem

>>> Single Run. Seeds: [1011, 1012, 1013]
Loading and splitting data (once for all seeds)...


Build train set: 100%|██████████████████████████████████████████████████████████████████| 10/10 [00:09<00:00,  1.03it/s]


Train samples: 2396, shape: (2396, 100, 3)
Test30 samples: 18000, shape: (18000, 100, 3)
Feature dimension: 3
Data dimension: 3
Ep 1/200 | Loss: 1.7108 | Acc: 44.28% | Mem: 2737.00MB | Time：23.57s
Ep 20/200 | Loss: 0.0918 | Acc: 97.69% | Mem: 2737.00MB | Time：4.14s
Ep 40/200 | Loss: 0.0331 | Acc: 99.14% | Mem: 2737.00MB | Time：4.14s
Ep 60/200 | Loss: 0.0102 | Acc: 99.67% | Mem: 2737.00MB | Time：4.13s
Ep 80/200 | Loss: 0.0042 | Acc: 99.92% | Mem: 2737.00MB | Time：4.14s
Ep 100/200 | Loss: 0.0004 | Acc: 100.00% | Mem: 2737.00MB | Time：4.15s
Ep 120/200 | Loss: 0.0002 | Acc: 100.00% | Mem: 2737.00MB | Time：4.15s
Ep 140/200 | Loss: 0.0001 | Acc: 100.00% | Mem: 2737.00MB | Time：4.14s
Ep 160/200 | Loss: 0.0001 | Acc: 100.00% | Mem: 2737.00MB | Time：4.13s
Ep 180/200 | Loss: 0.0001 | Acc: 100.00% | Mem: 2737.00MB | Time：4.12s
Ep 200/200 | Loss: 0.0001 | Acc: 100.00% | Mem: 2737.00MB | Time：4.15s

=== Training Stats ===
Total Time: 849.74s
Total Steps: 7600
Time per 1000 steps: 111.81s
Avg GPU Me

>>> Single Run. Seeds: [1011, 1012, 1013]
Loading and splitting data (once for all seeds)...


Build train set: 100%|██████████████████████████████████████████████████████████████████| 52/52 [02:15<00:00,  2.61s/it]


Train samples: 12401, shape: (12401, 100, 3)
Test30 samples: 93510, shape: (93510, 100, 3)
Feature dimension: 3
Data dimension: 3
Ep 1/200 | Loss: 3.0813 | Acc: 17.94% | Mem: 1717.00MB | Time：33.31s
Ep 20/200 | Loss: 0.3388 | Acc: 89.75% | Mem: 1717.00MB | Time：19.29s
Ep 40/200 | Loss: 0.0744 | Acc: 98.22% | Mem: 1717.00MB | Time：19.16s
Ep 60/200 | Loss: 0.0192 | Acc: 99.70% | Mem: 1717.00MB | Time：19.13s
Ep 80/200 | Loss: 0.0060 | Acc: 99.91% | Mem: 1717.00MB | Time：19.16s
Ep 100/200 | Loss: 0.0043 | Acc: 99.94% | Mem: 1716.00MB | Time：19.17s
Ep 120/200 | Loss: 0.0040 | Acc: 99.95% | Mem: 1716.00MB | Time：19.18s
Ep 140/200 | Loss: 0.0012 | Acc: 99.99% | Mem: 1716.00MB | Time：19.29s
Ep 160/200 | Loss: 0.0005 | Acc: 100.00% | Mem: 1716.00MB | Time：19.19s
Ep 180/200 | Loss: 0.0004 | Acc: 100.00% | Mem: 1716.00MB | Time：19.19s
Ep 200/200 | Loss: 0.0003 | Acc: 100.00% | Mem: 1716.00MB | Time：19.05s

=== Training Stats ===
Total Time: 3869.40s
Total Steps: 38800
Time per 1000 steps: 99.73s
